# Calibración de la sensibilidad del detector (Fase 6)

**Objetivo:** responder con datos a la pregunta *"¿cómo hago que el modelo detecte más daños sin llenarse de falsas alarmas?"*

**Qué hace**
1. Reconstruye **exactamente** los mismos conjuntos train/val/test de la Fase 4 (misma semilla y bloques; si cambias `COUNTRIES`, `SEED` o `BLOCK` ya no son los mismos).
2. Carga tu `best.pt` desde Drive y lo prueba con varias configuraciones de inferencia (umbral, resolución, TTA) sobre **validación**.
3. Mide con un evaluador propio la **precisión y el recall por clase** para cada umbral (IoU ≥ 0,5).
4. Elige la configuración y los umbrales por clase **sobre validación** y los confirma **una sola vez en test** (así la cifra final no está sesgada).
5. Te entrega la línea `PERFIL_SENSIBLE=...` para pegar en el `.env` de la aplicación.

**Cómo leer el resultado:** bajar el umbral siempre sube el recall y baja la precisión. Aquí no hay un valor "correcto": lo decides tú con `PRECISION_MIN` (celda 12), que es *cuántas falsas alarmas toleras*. Todo lo que se imprime es **medido** en tus datos de RDD2022; no se promete ninguna mejora de antemano.

**Límites:** se mide sobre RDD2022 (Japón, India, Chequia, EE. UU., China-moto), no sobre carreteras de Ecuador. Los milisegundos que se imprimen son de la GPU de Colab, **no** de tu computador.

**Requisitos:** GPU T4, tener `final_full/best.pt` en Drive (resultado de la Fase 4) y ~25 GB libres para volver a descargar el dataset. Ejecuta las celdas en orden.

## 1) Comprobar los recursos disponibles
Muestra GPU, RAM, CPU y disco libre. **Verificación:** debe aparecer una línea `GPU: ...` (por ejemplo, Tesla T4). Si dice que no hay GPU, revisa el paso 1 de arriba.

In [ ]:
import os, sys, shutil, subprocess, platform

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return (r.stdout + r.stderr).strip()

print("Python:", sys.version.split()[0], "|", platform.platform())

smi = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
SMI_OK = bool(smi) and not any(w in smi.lower() for w in ("not found", "failed", "no devices"))
print("GPU:", smi if SMI_OK else "NO se detectó GPU (activa una en el menú de Entorno de ejecución)")

try:
    with open("/proc/meminfo") as f:
        ram_gb = int(f.readline().split()[1]) / 1024**2
    print(f"RAM del sistema: {ram_gb:.1f} GB")
except Exception:
    pass
print("CPUs:", os.cpu_count())

WORK = "/content" if os.path.isdir("/content") else os.path.abspath("work")
os.makedirs(WORK, exist_ok=True)
free_gb = shutil.disk_usage(WORK).free / 1024**3
print(f"Disco libre en {WORK}: {free_gb:.1f} GB")
if free_gb < 20:
    print("AVISO: menos de 20 GB libres. La descarga alternativa de Figshare (13,3 GB) no cabría; "
          "la fuente original por países solo necesita ~5 GB.")

## 2) Instalar dependencias y verificar versiones
Instala `ultralytics` (trae PyTorch si hace falta). **Verificación:** se imprimen las versiones; **anótalas** (las usaremos en la documentación). Si más adelante quieres reproducir exactamente este entorno, fija esa versión con `%pip install ultralytics==X.Y.Z`.

In [ ]:
%pip install -q ultralytics==8.4.184 tqdm requests pandas matplotlib pyyaml pillow
import ultralytics, torch
ultralytics.checks()
print("ultralytics:", ultralytics.__version__)
print("torch:", torch.__version__, "| CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | VRAM: {p.total_memory / 1024**3:.1f} GB")

## 3) Configuración (edita solo esta celda)
- Aquí **no** se entrena: `MODE` debe quedarse en `"full"` para reconstruir el dataset completo igual que en la Fase 4.
- `COUNTRIES`: se excluyen a propósito **Norway** (9,9 GB) y **China_Drone** (vista de dron, solo train). Los tamaños de descarga provienen del repositorio oficial de RDD2022.
- `MODE`: ver tabla de la introducción.
- El split se hace por **bloques de imágenes consecutivas** (`BLOCK`) para que fotogramas casi idénticos no queden repartidos entre train y val/test.

In [ ]:
import torch

MODE = "full"         # no cambiar: la calibración usa el dataset completo
COUNTRIES = ["Japan", "India", "Czech", "United_States", "China_MotorBike"]
SEED = 0
BLOCK = 50            # imágenes consecutivas que viajan juntas al mismo split
VAL_FRAC, TEST_FRAC = 0.10, 0.10
BG_FRACTION = 0.10    # imágenes sin daño a conservar (fracción de las etiquetadas)
TIME_HOURS = 4.0      # presupuesto del entrenamiento completo (MODE="full")
CLS_PW = 0.0          # peso por frecuencia de clase; mira la celda 9 antes de cambiarlo
ECU_DIR = ""          # carpeta con TUS fotos de Ecuador (opcional, celda 16)
BASE_URL = "https://bigdatacup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022/Country_Specific_Data_CRDDC2022"

HAS_GPU = torch.cuda.is_available()
DEVICE = 0 if HAS_GPU else "cpu"
if MODE not in ("smoke", "benchmark", "full"):
    raise ValueError("MODE debe ser 'smoke', 'benchmark' o 'full'")
if MODE != "smoke" and not HAS_GPU:
    raise RuntimeError("Sin GPU solo se permite MODE='smoke'. Activa una GPU en 'Entorno de ejecución'.")

PROFILES = {
    "smoke":     dict(model="yolo26n.pt", imgsz=320, batch=16 if HAS_GPU else 8, limits=(200, 60, 60)),
    "benchmark": dict(model="yolo26s.pt", imgsz=640, batch=-1, limits=None),
    "full":      dict(model="yolo26s.pt", imgsz=640, batch=-1, limits=None, patience=20),
}
PROFILE = PROFILES[MODE]
print("MODE:", MODE, "|", PROFILE, "| device:", DEVICE)

## 4) Carpeta de resultados (Google Drive)
Los resultados se guardan en tu Drive para no perderlos si Colab se desconecta. Colab pide permiso la primera vez. **Verificación:** se imprime `OUT_DIR`.

In [ ]:
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/rdd2022_yolo26")
    IN_COLAB = True
except ImportError:
    OUT_DIR = Path(os.path.abspath("outputs"))
    IN_COLAB = False
OUT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = OUT_DIR / "runs"
print("IN_COLAB:", IN_COLAB, "| OUT_DIR:", OUT_DIR)

## 5) Descargar y descomprimir RDD2022
Intenta primero el servidor original (descarga por país, ~2,4 GB en total). **Si responde 403 o no está disponible, usa automáticamente Figshare**, que aloja el mismo dataset con licencia **CC BY 4.0** (verificada en su registro). Figshare solo ofrece **un único zip de 13,3 GB** con todos los países; el cuaderno lo descarga (con reanudación y verificación MD5), extrae **solo** los países de `COUNTRIES` y borra el zip grande para liberar disco. Necesita unos **20 GB libres**. **Verificación:** al final cada país muestra `OK`.

In [ ]:
import requests, zipfile, hashlib
from tqdm.auto import tqdm

RAW = Path(WORK) / "rdd2022_raw"
RAW.mkdir(parents=True, exist_ok=True)

# Fuente alternativa (registro de Figshare, verificado): un único zip con todos los países
FIGSHARE_URL = "https://ndownloader.figshare.com/files/38030910"
FIGSHARE_SIZE = 13264172619
FIGSHARE_MD5 = "b62bd51d2ffcfaa76c60f234f0cc2bb3"
VERIFY_MD5 = True
DELETE_BIG_ZIP = True

def download(url, dest, expected=0, chunk=1 << 20, retries=5):
    dest = Path(dest)
    part = Path(str(dest) + ".part")
    if dest.exists() and (not expected or dest.stat().st_size == expected):
        print("ya descargado:", dest.name)
        return
    for attempt in range(1, retries + 1):
        done = part.stat().st_size if part.exists() else 0
        if expected and done == expected:
            break
        headers = {"Range": f"bytes={done}-"} if done else {}
        try:
            with requests.get(url, stream=True, headers=headers, timeout=60) as r:
                if r.status_code == 416:      # rango inválido: empezamos de cero
                    part.unlink(missing_ok=True)
                    continue
                r.raise_for_status()
                if done and r.status_code != 206:
                    done = 0                  # el servidor ignoró Range: empezamos de cero
                total = done + int(r.headers.get("Content-Length", 0))
                with open(part, "ab" if done else "wb") as f, tqdm(
                        total=total or None, initial=done, unit="B", unit_scale=True, desc=dest.name) as bar:
                    for blk in r.iter_content(chunk):
                        f.write(blk)
                        bar.update(len(blk))
            break
        except requests.HTTPError as e:
            if e.response is not None and e.response.status_code in (401, 403, 404):
                raise
            print(f"intento {attempt}/{retries} falló: {e}")
            if attempt == retries:
                raise
        except requests.RequestException as e:
            print(f"intento {attempt}/{retries} falló: {e}")
            if attempt == retries:
                raise
    size = part.stat().st_size
    if expected and size != expected:
        raise IOError(f"Descarga incompleta de {dest.name}: {size} de {expected} bytes. "
                      "Vuelve a ejecutar la celda para reanudar.")
    part.replace(dest)

def md5_file(path, n=1 << 22):
    h = hashlib.md5()
    total = Path(path).stat().st_size
    with open(path, "rb") as f, tqdm(total=total, unit="B", unit_scale=True, desc="verificando MD5") as bar:
        for blk in iter(lambda: f.read(n), b""):
            h.update(blk)
            bar.update(len(blk))
    return h.hexdigest()

def extract_country_from_big(zpath, c):
    out = RAW / c
    key = c.lower()
    with zipfile.ZipFile(zpath) as z:
        names = z.namelist()
        files = [n for n in names if not n.endswith("/")]
        # archivos cuya ruta (carpeta o nombre) menciona el país, p. ej. RDD2022_Japan/... o Japan/...
        members = [n for n in files if not n.lower().endswith(".zip")
                   and any(key in part.lower() for part in Path(n).parts)]
        nested = [n for n in files if n.lower().endswith(".zip") and key in Path(n).name.lower()]
        if members:
            for n in tqdm(members, desc=f"extrayendo {c}"):
                z.extract(n, out)
        elif nested:
            tmp = RAW / "_nested"
            for n in nested:
                z.extract(n, tmp)
                with zipfile.ZipFile(tmp / n) as zz:
                    zz.extractall(out)
            shutil.rmtree(tmp, ignore_errors=True)
        else:
            second = sorted({"/".join(Path(n).parts[:3]) for n in names if n})[:40]
            raise RuntimeError(f"No encontré '{c}' dentro del zip. Rutas de hasta 3 niveles: {second}")
    (out / ".extracted").touch()

missing = [c for c in COUNTRIES if not (RAW / c / ".extracted").exists()]
if not missing:
    print("Todos los países ya estaban extraídos.")
else:
    first_url = f"{BASE_URL}/RDD2022_{missing[0]}.zip"
    try:
        s3_ok = requests.head(first_url, allow_redirects=True, timeout=30).status_code == 200
    except requests.RequestException:
        s3_ok = False
    print("Fuente:", "servidor original" if s3_ok else
          "Figshare (el servidor original no respondió 200: se usa la alternativa)")
    if s3_ok:
        for c in missing:
            zpath = RAW / f"RDD2022_{c}.zip"
            download(f"{BASE_URL}/RDD2022_{c}.zip", zpath)
            if not zipfile.is_zipfile(zpath):
                raise IOError(f"{zpath.name} no es un zip válido; bórralo y vuelve a ejecutar la celda")
            with zipfile.ZipFile(zpath) as z:
                z.extractall(RAW / c)
            (RAW / c / ".extracted").touch()
    else:
        free = shutil.disk_usage(WORK).free
        big = RAW / "RDD2022_released_through_CRDDC2022.zip"
        if not big.exists() and free < FIGSHARE_SIZE + 4e9:
            raise RuntimeError(f"Disco insuficiente: hay {free / 1e9:.1f} GB libres y se necesitan unos "
                               f"{(FIGSHARE_SIZE + 4e9) / 1e9:.0f} GB. Reduce COUNTRIES no ayuda con esta fuente: "
                               "usa un entorno con más disco.")
        download(FIGSHARE_URL, big, expected=FIGSHARE_SIZE)
        if VERIFY_MD5:
            got = md5_file(big)
            if got != FIGSHARE_MD5:
                raise IOError(f"MD5 distinto (esperado {FIGSHARE_MD5}, obtenido {got}). Borra el zip y repite.")
            print("MD5 verificado.")
        for c in missing:
            extract_country_from_big(big, c)
        if DELETE_BIG_ZIP:
            big.unlink()
            print("Zip grande borrado para liberar disco.")
for c in COUNTRIES:
    print(c, "OK" if (RAW / c / ".extracted").exists() else "FALTA")

## 6) Revisar la estructura de carpetas y las anotaciones
No suponemos la estructura interna de los zip: la mostramos y el código de conversión la recorre de forma recursiva. **Verificación:** cada país debe tener imágenes `.jpg` y archivos `.xml` (las de *test* traen solo imágenes, sin xml).

In [ ]:
from collections import Counter
from PIL import Image
import random

IMG_EXT = {".jpg", ".jpeg", ".png"}

def tree(root, depth=3, prefix=""):
    if depth < 0:
        return
    for p in sorted(root.iterdir())[:8]:
        if p.name.startswith("."):
            continue
        print(prefix + p.name + ("/" if p.is_dir() else ""))
        if p.is_dir():
            tree(p, depth - 1, prefix + "   ")

for c in COUNTRIES:
    root = RAW / c
    files = [p for p in root.rglob("*") if p.is_file()]
    n_img = sum(p.suffix.lower() in IMG_EXT for p in files)
    n_xml = sum(p.suffix.lower() == ".xml" for p in files)
    print(f"\n=== {c}: {n_img} imágenes | {n_xml} xml ===")
    tree(root, depth=2)

first_xml = next(RAW.rglob("*.xml"), None)
if first_xml:
    print("\nEjemplo de anotación:", first_xml)
    print(first_xml.read_text()[:900])

rng_ = random.Random(0)
sizes = Counter()
for c in COUNTRIES:
    imgs = [p for p in (RAW / c).rglob("*") if p.suffix.lower() in IMG_EXT]
    for p in rng_.sample(imgs, min(60, len(imgs))):
        with Image.open(p) as im:
            sizes[(c, im.size)] += 1
print("\nTamaños de imagen (muestra de hasta 60 por país):")
for (c, s), n in sorted(sizes.items()):
    print(f"  {c:16s} {s[0]}x{s[1]}  x{n}")

## 7) Convertir Pascal VOC → YOLO, limpiar y deduplicar
- Clases usadas: `D00` fisura longitudinal, `D10` fisura transversal, `D20` piel de cocodrilo, `D40` bache (códigos documentados por RDD2022). Cualquier otro nombre de clase se **cuenta y se descarta** (verás el reporte).
- Se usa el tamaño **real** de la imagen (no el del xml) para normalizar; las cajas fuera de imagen se recortan y las degeneradas se descartan.
- Se eliminan duplicados **exactos** (mismo contenido, por hash MD5).
- Se conserva una fracción limitada de imágenes sin daño (`BG_FRACTION`).

**Verificación:** lee el reporte; los contadores de “descartado” no deberían ser una parte grande del total.

In [ ]:
import re, hashlib, json
import xml.etree.ElementTree as ET

CLASS_CODES = ["D00", "D10", "D20", "D40"]
CLASS_NAMES = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole"]
CODE2ID = {c: i for i, c in enumerate(CLASS_CODES)}

def natural_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

def md5(path, n=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for blk in iter(lambda: f.read(n), b""):
            h.update(blk)
    return h.hexdigest()

stats = Counter()
unknown = Counter()
records = []

for c in COUNTRIES:
    root = RAW / c
    imgs = {}
    for p in root.rglob("*"):
        if p.suffix.lower() in IMG_EXT:
            if p.stem in imgs:
                stats["nombre_de_imagen_repetido"] += 1
                continue
            imgs[p.stem] = p
    xmls = sorted(root.rglob("*.xml"), key=lambda p: natural_key(p.stem))
    labeled_stems = set()
    for x in tqdm(xmls, desc=f"convirtiendo {c}"):
        try:
            tr = ET.parse(x).getroot()
        except ET.ParseError:
            stats["xml_ilegible"] += 1
            continue
        stem = x.stem
        if stem not in imgs:
            fn = tr.findtext("filename")
            stem = Path(fn).stem if fn and Path(fn).stem in imgs else None
        if stem is None:
            stats["xml_sin_imagen"] += 1
            continue
        try:
            with Image.open(imgs[stem]) as im:
                W, H = im.size
        except Exception:
            stats["imagen_corrupta"] += 1
            continue
        sz = tr.find("size")
        if sz is not None:
            try:
                if (int(float(sz.findtext("width"))), int(float(sz.findtext("height")))) != (W, H):
                    stats["tamano_xml_distinto_de_imagen"] += 1
            except (TypeError, ValueError):
                pass
        labels = []
        for ob in tr.findall("object"):
            name = (ob.findtext("name") or "").strip()
            if name not in CODE2ID:
                unknown[name] += 1
                continue
            bb = ob.find("bndbox")
            try:
                x1, y1, x2, y2 = (float(bb.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
            except (TypeError, ValueError, AttributeError):
                stats["caja_ilegible"] += 1
                continue
            cx1, cy1, cx2, cy2 = max(0.0, x1), max(0.0, y1), min(float(W), x2), min(float(H), y2)
            if (cx1, cy1, cx2, cy2) != (x1, y1, x2, y2):
                stats["caja_recortada"] += 1
            if cx2 - cx1 < 2 or cy2 - cy1 < 2:
                stats["caja_degenerada_descartada"] += 1
                continue
            labels.append((CODE2ID[name], (cx1 + cx2) / 2 / W, (cy1 + cy2) / 2 / H, (cx2 - cx1) / W, (cy2 - cy1) / H))
        records.append(dict(country=c, stem=stem, img=str(imgs[stem]), labels=labels, W=W, H=H))
        labeled_stems.add(stem)
    stats[f"{c}: imagenes_sin_xml (test sin etiquetas)"] = len(imgs) - len(labeled_stems)

# duplicados exactos
seen, unique = {}, []
for r in tqdm(records, desc="buscando duplicados exactos"):
    h = md5(r["img"])
    if h in seen:
        stats["duplicado_exacto_descartado"] += 1
        continue
    r["md5"] = h
    seen[h] = r["stem"]
    unique.append(r)

# fondo limitado
labeled = [r for r in unique if r["labels"]]
bg = [r for r in unique if not r["labels"]]
rng = random.Random(SEED)
rng.shuffle(bg)
keep_bg = bg[: int(BG_FRACTION * len(labeled))]
stats["fondo_descartado_por_limite"] = len(bg) - len(keep_bg)
kept = labeled + keep_bg

print(f"\nImágenes con xml: {len(records)} | únicas: {len(unique)} | conservadas: {len(kept)} "
      f"({len(labeled)} con daño + {len(keep_bg)} sin daño)")
print("\nReporte de limpieza:")
for k, v in sorted(stats.items()):
    print(f"  {k}: {v}")
print("\nClases NO usadas (nombre -> cajas):", dict(unknown) if unknown else "ninguna")

## 8) Crear train / val / test sin fuga de datos
Dentro de cada país se ordenan las imágenes por número, se agrupan en bloques de `BLOCK` consecutivas y se reparten **bloques completos** entre splits. No garantiza cero fuga (no conocemos el orden real de captura), pero reduce mucho el riesgo de que fotogramas vecinos queden a ambos lados. El split de *test* original no tiene etiquetas, así que **creamos el nuestro**.

Las imágenes se enlazan con *hardlinks* (sin duplicar disco) y, si no es posible, se copian. **Verificación:** se imprime el tamaño de cada split.

In [ ]:
DS = Path(WORK) / "rdd2022_yolo"
if DS.exists():
    shutil.rmtree(DS)
for s in ("train", "val", "test"):
    (DS / "images" / s).mkdir(parents=True)
    (DS / "labels" / s).mkdir(parents=True)

split_of = {}
for c in COUNTRIES:
    recs = sorted([r for r in kept if r["country"] == c], key=lambda r: natural_key(r["stem"]))
    blocks = [recs[i:i + BLOCK] for i in range(0, len(recs), BLOCK)]
    rng.shuffle(blocks)
    n = len(blocks)
    n_val = max(1, round(n * VAL_FRAC)) if n >= 3 else 0
    n_test = max(1, round(n * TEST_FRAC)) if n >= 3 else 0
    for i, b in enumerate(blocks):
        s = "val" if i < n_val else "test" if i < n_val + n_test else "train"
        for r in b:
            split_of[id(r)] = s

if PROFILE["limits"]:
    lim = dict(zip(("train", "val", "test"), PROFILE["limits"]))
    cnt = Counter()
    for r in rng.sample(kept, len(kept)):          # aleatorio: así smoke incluye todos los países
        s = split_of[id(r)]
        cnt[s] += 1
        if cnt[s] > lim[s]:
            split_of[id(r)] = None
    print("SMOKE: se recortan los splits a", lim)

used_names, manifest = set(), []
for r in tqdm(kept, desc="materializando dataset"):
    s = split_of[id(r)]
    if s is None:
        continue
    base = r["stem"] if r["stem"].startswith(r["country"]) else f"{r['country']}_{r['stem']}"
    name, k = base, 1
    while name in used_names:
        name, k = f"{base}_{k}", k + 1
    used_names.add(name)
    src = Path(r["img"])
    dst = DS / "images" / s / (name + src.suffix.lower())
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)
    with open(DS / "labels" / s / (name + ".txt"), "w") as f:
        for cid, cx, cy, w, h in r["labels"]:
            f.write(f"{cid} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")
    manifest.append(dict(split=s, country=r["country"], name=name, n_boxes=len(r["labels"]), md5=r["md5"],
                         width=r["W"], height=r["H"]))

import pandas as pd
manifest_df = pd.DataFrame(manifest)
(OUT_DIR / "calibracion").mkdir(exist_ok=True)
manifest_df.to_csv(OUT_DIR / "calibracion" / "manifest_recreado.csv", index=False)  # no pisa el de la Fase 4
print(manifest_df.groupby("split").size().to_string())

## 9) Estadísticas del dataset y comprobaciones anti-fuga
Cuenta cajas por clase y split, y comprueba que **ninguna imagen (ni contenido idéntico) aparece en dos splits**. **Verificación:** deben pasar los `assert`. Si una clase está muy desbalanceada, la celda lo avisa.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

rows = []
for s in ("train", "val", "test"):
    cnt = Counter()
    for t in (DS / "labels" / s).glob("*.txt"):
        for line in t.read_text().splitlines():
            cnt[int(line.split()[0])] += 1
    rows.append({"split": s, **{CLASS_NAMES[i]: cnt.get(i, 0) for i in range(4)}})
class_df = pd.DataFrame(rows).set_index("split")
display(class_df)
display(manifest_df.pivot_table(index="country", columns="split", values="name", aggfunc="count", fill_value=0))

class_df.T.plot(kind="bar", figsize=(8, 3.5), title="Cajas por clase y split")
plt.tight_layout()
plt.show()

# --- comprobaciones anti-fuga
assert manifest_df["name"].is_unique, "Nombres repetidos entre splits"
dup_hash = manifest_df.groupby("md5")["split"].nunique()
assert (dup_hash <= 1).all(), "Hay contenido idéntico en más de un split"
print("OK: sin nombres repetidos y sin contenido idéntico entre splits.")

tr = class_df.loc["train"]
nonzero = tr[tr > 0]
if len(nonzero) < 4:
    print("AVISO: alguna clase no tiene ejemplos en train:", list(tr[tr == 0].index))
elif nonzero.max() / nonzero.min() > 10:
    print(f"AVISO: desbalance {nonzero.max() / nonzero.min():.0f}:1 entre clases. "
          "Entrena primero con CLS_PW=0.0 y, si la clase rara rinde mal, prueba CLS_PW=0.25.")
else:
    print(f"Desbalance máx/mín en train: {nonzero.max() / nonzero.min():.1f}:1 (moderado).")

## 9b) Verificar que train/val/test son los MISMOS de la Fase 4
`best.pt` se entrenó con las imágenes de *train*. Si el split reconstruido no fuera idéntico, imágenes que el modelo ya vio podrían caer en *val* o *test* y las métricas saldrían infladas. Se compara con el `manifest_full.csv` que guardó la Fase 4 en tu Drive (por huella MD5 de cada imagen). **Si no coincide, el cuaderno se detiene.**

In [ ]:
import pandas as pd
M_FASE4 = OUT_DIR / "manifest_full.csv"
assert M_FASE4.exists(), (f"No encuentro {M_FASE4} (el manifiesto de la Fase 4). Sin él no puedo comprobar que el split es el mismo; "
                          "si lo renombraste, ajusta M_FASE4.")
viejo = pd.read_csv(M_FASE4)
a = set(zip(viejo["md5"], viejo["split"]))
b = set(zip(manifest_df["md5"], manifest_df["split"]))
print(f"Fase 4: {len(a)} imágenes | reconstruido: {len(b)} imágenes | iguales: {len(a & b)}")
solo_a, solo_b = a - b, b - a
assert not solo_a and not solo_b, (f"El split NO coincide con el de la Fase 4 ({len(solo_a)} solo en la Fase 4, {len(solo_b)} solo ahora). "
                                   "No sigas: las métricas podrían estar infladas. Revisa COUNTRIES, SEED y BLOCK.")
print("✅ El split reconstruido es idéntico al de la Fase 4 (mismas imágenes en train, val y test).")

## 10) Archivo YAML del dataset
Ultralytics lo usa para encontrar imágenes, etiquetas y nombres de clase. **Verificación:** se vuelve a leer y se imprime.

In [ ]:
import yaml
DATA_YAML = DS / "data.yaml"
cfg_yaml = {
    "path": str(DS),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: n for i, n in enumerate(CLASS_NAMES)},
}
DATA_YAML.write_text(yaml.safe_dump(cfg_yaml, sort_keys=False))
print(DATA_YAML.read_text())
assert yaml.safe_load(DATA_YAML.read_text())["names"][3] == "pothole"

## 11) Cargar tu modelo entrenado
Busca `best.pt` en la carpeta que dejó la Fase 4. Si lo moviste, cambia `MODEL_PT`. **Verificación:** imprime su SHA-256 (debe coincidir con el de tu `run_info.json`: `59e030a0…8d88` en tu entrenamiento largo).

In [ ]:
import hashlib
from ultralytics import YOLO

MODEL_PT = OUT_DIR / "final_full" / "best.pt"      # <- cámbialo si lo guardaste en otro sitio
assert MODEL_PT.exists(), f"No encuentro {MODEL_PT}. Ajusta MODEL_PT."
h = hashlib.sha256(MODEL_PT.read_bytes()).hexdigest()
print("best.pt:", MODEL_PT, "| SHA-256:", h)
probe = YOLO(str(MODEL_PT))
print("Clases del modelo:", probe.names)
assert [probe.names[i] for i in range(4)] == CLASS_NAMES, "Las clases del modelo no coinciden con las del dataset"
del probe
CAL_DIR = OUT_DIR / "calibracion"; CAL_DIR.mkdir(exist_ok=True)

## 12) Parámetros de la calibración (esto lo decides tú)
- `PRECISION_MIN`: precisión mínima que exiges **por clase** al perfil sensible. `0.50` significa "como mucho 1 de cada 2 detecciones puede ser falsa alarma". Más bajo → detecta más, pero molesta más. No hay un valor universal; empieza con 0,50 y mira la tabla.
- `MIN_GAIN`: una configuración más cara (más resolución o TTA) solo se elige si mejora el recall medio al menos esto (en puntos de 0 a 1). Evita pagar 2–3× de tiempo por una mejora despreciable.
- `QUICK`: pon un número (p. ej. `600`) para evaluar solo esa cantidad de imágenes de validación y tener una primera idea rápida. `None` = todas (recomendado para decidir).

In [ ]:
PRECISION_MIN = 0.50
MIN_GAIN = 0.02
QUICK = None          # p. ej. 600 para una pasada rápida
IOU_MATCH = 0.5       # una detección acierta si IoU >= 0.5 con una etiqueta de su misma clase

# Configuraciones, de la más barata a la más cara en tiempo de CPU. El orden importa para la elección.
CONFIGS = [
    dict(nombre="base 640",   imgsz=640, augment=False),
    dict(nombre="imgsz 800",  imgsz=800, augment=False),
    dict(nombre="imgsz 960",  imgsz=960, augment=False),
    dict(nombre="TTA 640",    imgsz=640, augment=True),
    dict(nombre="TTA 800",    imgsz=800, augment=True),
    # solo informativa: la aplicación no la puede usar (cabeza sin NMS del modelo)
    dict(nombre="cabeza sin NMS (informativa)", imgsz=640, augment=False, extra=dict(nms=False), info=True),
]
print(len(CONFIGS), "configuraciones")

## 13) Evaluador propio
Ultralytics reporta precisión/recall solo en un punto interno de su curva. Para saber qué pasa con *cada* umbral necesitamos nuestro propio recuento:

- Cada detección se empareja, en orden de confianza, con la etiqueta **de su misma clase** no usada con mayor IoU (≥ 0,5). Si no hay, es falsa alarma (FP). Las etiquetas sin emparejar son daños no detectados (FN).
- Como el emparejamiento va por orden de confianza, **subir o bajar el umbral solo añade o quita las detecciones de menor confianza**: basta emparejar una vez con confianza ≥ 0,01 y luego filtrar.
- Se calcula también AP50 (misma interpolación de 101 puntos que Ultralytics) y, más abajo, se compara con `model.val()` como **verificación cruzada**.

In [ ]:
import time, json, gc as recolector
import torch
import numpy as np
from pathlib import Path

NC = len(CLASS_NAMES)
THRS = np.round(np.arange(0.02, 0.91, 0.01), 2)
_trapz = getattr(np, "trapezoid", None) or np.trapz


def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0]); iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2]); iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(ix2 - ix1, 0, None) * np.clip(iy2 - iy1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1]); ab = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / (aa[:, None] + ab[None, :] - inter + 1e-9)


def leer_gt(path, w, h):
    # etiquetas YOLO normalizadas -> (clases, cajas xyxy en píxeles)
    p = Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return np.zeros(0, int), np.zeros((0, 4))
    a = np.loadtxt(p, ndmin=2)
    cls = a[:, 0].astype(int)
    xc, yc, bw, bh = a[:, 1] * w, a[:, 2] * h, a[:, 3] * w, a[:, 4] * h
    return cls, np.stack([xc - bw / 2, yc - bh / 2, xc + bw / 2, yc + bh / 2], 1)


def emparejar(conf, cls, boxes, gt_cls, gt_boxes):
    # devuelve tp[i] (bool) para cada predicción, con emparejamiento voraz por confianza y por clase
    tp = np.zeros(len(conf), bool)
    order = np.argsort(-conf, kind="stable")
    for c in np.unique(cls):
        gi = np.where(gt_cls == c)[0]
        if len(gi) == 0:
            continue
        pi = order[cls[order] == c]
        M = iou_matrix(boxes[pi], gt_boxes[gi])
        usado = np.zeros(len(gi), bool)
        for r, p in enumerate(pi):
            cand = np.where(~usado & (M[r] >= IOU_MATCH))[0]
            if len(cand):
                j = cand[np.argmax(M[r][cand])]
                usado[j] = True
                tp[p] = True
    return tp


def predecir(rutas, imgsz, augment=False, extra=None):
    # Una imagen a la vez (evita CUDA OOM en Colab) y modelo NUEVO por configuración:
    # Ultralytics conserva estado entre llamadas (p. ej. nms=False).
    modelo = YOLO(str(MODEL_PT))
    out, t0 = {}, time.time()
    with torch.inference_mode():
        for p in rutas:
            r = modelo.predict(source=str(p), conf=0.01, imgsz=imgsz, augment=augment, device=DEVICE,
                               verbose=False, max_det=300, **(extra or {}))[0]
            out[Path(r.path).stem] = dict(
                boxes=r.boxes.xyxy.cpu().numpy(), conf=r.boxes.conf.cpu().numpy(),
                cls=r.boxes.cls.cpu().numpy().astype(int), hw=r.orig_shape)
    ms = (time.time() - t0) / max(1, len(rutas)) * 1000
    del modelo
    recolector.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out, ms


def contar(preds, labels_dir):
    # agrega, por clase: confianzas, aciertos (tp) y nº de etiquetas
    d = {c: dict(conf=[], tp=[], n_gt=0) for c in range(NC)}
    for stem, p in preds.items():
        h, w = p["hw"]
        gc, gb = leer_gt(Path(labels_dir) / (stem + ".txt"), w, h)
        for c in range(NC):
            d[c]["n_gt"] += int((gc == c).sum())
        if len(p["conf"]):
            tp = emparejar(p["conf"], p["cls"], p["boxes"], gc, gb)
            for c in range(NC):
                m = p["cls"] == c
                d[c]["conf"].append(p["conf"][m]); d[c]["tp"].append(tp[m])
    for c in range(NC):
        d[c]["conf"] = np.concatenate(d[c]["conf"]) if d[c]["conf"] else np.zeros(0)
        d[c]["tp"] = np.concatenate(d[c]["tp"]) if d[c]["tp"] else np.zeros(0, bool)
    return d


def punto(dc, t):
    m = dc["conf"] >= t
    tp = int(dc["tp"][m].sum()); fp = int(m.sum()) - tp; fn = dc["n_gt"] - tp
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / dc["n_gt"] if dc["n_gt"] else 0.0
    F = 2 * P * R / (P + R) if P + R else 0.0
    return dict(P=P, R=R, F1=F, TP=tp, FP=fp, FN=fn)


def ap50(dc):
    if dc["n_gt"] == 0:
        return float("nan")          # clase sin etiquetas: no entra en los promedios
    if len(dc["conf"]) == 0:
        return 0.0
    o = np.argsort(-dc["conf"], kind="stable")
    tpc = np.cumsum(dc["tp"][o]); fpc = np.cumsum(~dc["tp"][o])
    rec = tpc / (dc["n_gt"] + 1e-16); prec = tpc / (tpc + fpc)
    mrec = np.concatenate(([0.0], rec, [1.0])); mpre = np.concatenate(([1.0], prec, [0.0]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    return float(_trapz(np.interp(x, mrec, mpre), x))


def umbral_para_precision(dc, pmin):
    # umbral MÁS BAJO (= más recall) cuya precisión sigue siendo >= pmin; None si ninguno lo logra
    for t in THRS:
        q = punto(dc, t)
        if q["TP"] + q["FP"] > 0 and q["P"] >= pmin:
            return float(t)
    return None


def media(tabla, col):
    # promedio por clase SOLO sobre las clases que tienen etiquetas
    return float(tabla.loc[tabla.etiquetas > 0, col].mean())


def mejor_f1(dc):
    return float(THRS[int(np.argmax([punto(dc, t)["F1"] for t in THRS]))])
print("Evaluador listo.")

## 14) Autoprueba del evaluador (casos con respuesta conocida)
Antes de confiar en las cifras, el evaluador debe acertar casos hechos a mano. Si alguna comprobación falla, el cuaderno se detiene.

In [ ]:
# Caso 1: una etiqueta clase 0 en [0,0,10,10]; 3 predicciones: acierto (0.9), duplicado (0.8), falsa alarma lejana (0.3)
gc, gb = np.array([0]), np.array([[0., 0., 10., 10.]])
boxes = np.array([[0, 0, 10, 10], [0, 0, 10, 10], [50, 50, 60, 60]], float)
conf = np.array([0.9, 0.8, 0.3]); cls = np.array([0, 0, 0])
tp = emparejar(conf, cls, boxes, gc, gb)
assert tp.tolist() == [True, False, False], tp          # el duplicado NO cuenta como acierto
d = {c: dict(conf=np.zeros(0), tp=np.zeros(0, bool), n_gt=0) for c in range(NC)}
d[0] = dict(conf=conf, tp=tp, n_gt=1)
q = punto(d[0], 0.5); assert (q["TP"], q["FP"], q["FN"]) == (1, 1, 0), q
q = punto(d[0], 0.85); assert (q["TP"], q["FP"], q["P"], q["R"]) == (1, 0, 1.0, 1.0), q
assert abs(ap50(d[0]) - 0.995) < 1e-6   # un detector perfecto da 0.995 (artefacto del borde de la interpolación de 101 puntos, igual que en Ultralytics)
# Caso 2: clase equivocada no acierta aunque la caja coincida
assert emparejar(np.array([0.9]), np.array([1]), np.array([[0, 0, 10, 10.]]), gc, gb).tolist() == [False]
# Caso 3: IoU justo por debajo de 0.5 no acierta; por encima sí
assert emparejar(np.array([0.9]), np.array([0]), np.array([[0, 0, 10, 4.9]]), gc, gb).tolist() == [False]
assert emparejar(np.array([0.9]), np.array([0]), np.array([[0, 0, 10, 5.1]]), gc, gb).tolist() == [True]
# Caso 4: dos etiquetas, dos predicciones, ninguna se reutiliza
gc2, gb2 = np.array([0, 0]), np.array([[0, 0, 10, 10.], [20, 0, 30, 10.]])
assert emparejar(np.array([0.9, 0.8]), np.array([0, 0]), np.array([[0, 0, 10, 10.], [0, 0, 10, 10.]]), gc2, gb2).tolist() == [True, False]
# Caso 5: umbral para precisión mínima
assert umbral_para_precision(d[0], 0.9) is not None and umbral_para_precision(d[0], 0.9) > 0.3
print("Autoprueba del evaluador: OK")

## 15) Probar todas las configuraciones en validación
Cada configuración recorre las imágenes de validación **una vez** con confianza ≥ 0,01 (así queda registrado todo lo que el modelo "ve") y luego se filtra por umbral sin volver a correr el modelo. Los tiempos que verás son de la GPU de Colab. **Puede tardar decenas de minutos** (las de TTA y 960 son las más lentas); si se corta, vuelve a ejecutar desde esta celda.

In [ ]:
import random

def lista_imagenes(split):
    imgs = sorted((DS / "images" / split).iterdir())
    imgs = [p for p in imgs if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
    return imgs

VAL_IMGS = lista_imagenes("val")
if QUICK:
    VAL_IMGS = random.Random(SEED).sample(VAL_IMGS, min(QUICK, len(VAL_IMGS)))
print("Imágenes de validación:", len(VAL_IMGS), "(QUICK)" if QUICK else "(todas)")
VAL_LABELS = DS / "labels" / "val"

RES = {}     # nombre -> dict(cfg, ms, ap50[clase], conteo[clase])
for cfg in CONFIGS:
    t0 = time.time()
    preds, ms = predecir(VAL_IMGS, cfg["imgsz"], cfg["augment"], cfg.get("extra"))
    d = contar(preds, VAL_LABELS)
    aps = [ap50(d[c]) for c in range(NC)]
    RES[cfg["nombre"]] = dict(cfg=cfg, ms=ms, d=d, aps=aps)
    print(f"{cfg['nombre']:34s} | mAP50 propio = {np.nanmean(aps):.4f} | {ms:6.1f} ms/img (GPU Colab) | {time.time()-t0:5.0f} s")
    del preds

## 16) Verificación cruzada contra Ultralytics
Mi evaluador y `model.val()` deben dar un mAP50 parecido en la configuración base (ambos con IoU 0,5). Una diferencia de hasta ~0,03 es normal (Ultralytics empareja distinto y usa letterbox rectangular); si es mayor, **no uses los resultados** y avísame.

In [ ]:
m = YOLO(str(MODEL_PT))
r = m.val(data=str(DATA_YAML), split="val", imgsz=640, batch=16, conf=0.001, plots=False, verbose=False, device=DEVICE)
mio = float(np.nanmean(RES["base 640"]["aps"]))
print(f"mAP50 Ultralytics (todo val): {r.box.map50:.4f} | mAP50 mío (config base): {mio:.4f} | diferencia: {abs(r.box.map50-mio):.4f}")
if QUICK:
    print("AVISO: con QUICK el conjunto es distinto; la comparación es solo orientativa.")
elif abs(r.box.map50 - mio) > 0.03:
    print("⚠️  DIFERENCIA > 0.03: no uses los resultados todavía y avísame.")
else:
    print("✅ Coinciden (diferencia <= 0.03; diferencias pequeñas son normales porque el emparejamiento no es idéntico).")
del m

## 17) Resultados en validación: ¿cuánto recall se gana y a qué precio?
Para cada configuración y clase se muestra el punto **normal** (umbral 0,25, el valor por defecto de la app) y el punto **sensible** (el umbral más bajo cuya precisión sigue ≥ `PRECISION_MIN`).

In [ ]:
import pandas as pd

def puntos_config(res, pmin):
    d = res["d"]; filas = []
    for c in range(NC):
        if d[c]["n_gt"] == 0:        # sin etiquetas no se puede calibrar: se deja el umbral normal
            filas.append(dict(clase=CLASS_NAMES[c], etiquetas=0, AP50=float("nan"), P_normal=float("nan"), R_normal=float("nan"),
                              umbral_sensible=0.25, P_sensible=float("nan"), R_sensible=float("nan"), dR=float("nan"), no_alcanza_pmin=True))
            continue
        t_s = umbral_para_precision(d[c], pmin)
        fallback = t_s is None
        if fallback:
            t_s = mejor_f1(d[c])
        n, s = punto(d[c], 0.25), punto(d[c], t_s)
        filas.append(dict(clase=CLASS_NAMES[c], etiquetas=d[c]["n_gt"], AP50=round(res["aps"][c], 3),
                          P_normal=round(n["P"], 3), R_normal=round(n["R"], 3),
                          umbral_sensible=t_s, P_sensible=round(s["P"], 3), R_sensible=round(s["R"], 3),
                          dR=round(s["R"] - n["R"], 3), no_alcanza_pmin=fallback))
    return pd.DataFrame(filas)

TABLAS = {nombre: puntos_config(res, PRECISION_MIN) for nombre, res in RES.items()}
for nombre, t in TABLAS.items():
    print(f"\n=== {nombre} | {RES[nombre]['ms']:.1f} ms/img (GPU Colab) | PRECISION_MIN={PRECISION_MIN} ===")
    print(t.to_string(index=False))
    print(f"recall medio: normal {media(t,'R_normal'):.3f} -> sensible {media(t,'R_sensible'):.3f} | precisión media sensible {media(t,'P_sensible'):.3f}")
    if t.no_alcanza_pmin.any():
        print("  (marcadas con no_alcanza_pmin=True: la clase no llega a PRECISION_MIN con ningún umbral; se usó el de mejor F1)")

## 18) Elegir la configuración
Se recorre la lista en orden de costo y **solo se cambia a una configuración más cara si el recall medio mejora al menos `MIN_GAIN`** respecto de la elegida hasta ese momento. La configuración "informativa" nunca se elige (la aplicación no puede usarla).

In [ ]:
elegida = None
for cfg in CONFIGS:
    if cfg.get("info"):
        continue
    t = TABLAS[cfg["nombre"]]
    if elegida is None or media(t, "R_sensible") >= media(TABLAS[elegida["nombre"]], "R_sensible") + MIN_GAIN:
        elegida = cfg
tabla_e = TABLAS[elegida["nombre"]]
print("Configuración elegida:", elegida["nombre"])
print("Recall medio por configuración (sensible):")
for cfg in CONFIGS:
    t = TABLAS[cfg["nombre"]]
    print(f"  {cfg['nombre']:34s} {media(t,'R_sensible'):.3f}  ({media(t,'R_sensible')-media(TABLAS['base 640'],'R_sensible'):+.3f} vs base)  {RES[cfg['nombre']]['ms']:.1f} ms/img (GPU Colab)")
umbrales = {r.clase: float(r.umbral_sensible) for r in tabla_e.itertuples()}
print("Umbrales por clase:", umbrales)

## 19) Confirmación en test (una sola vez)
Los umbrales se eligieron con validación; **test no se usó para elegir nada**. Estas son las cifras que puedes citar. Se comparan el perfil normal (config base, umbral 0,25) y el sensible elegido.

In [ ]:
TEST_IMGS = lista_imagenes("test")
if QUICK:
    TEST_IMGS = random.Random(SEED).sample(TEST_IMGS, min(QUICK, len(TEST_IMGS)))
TEST_LABELS = DS / "labels" / "test"
print("Imágenes de test:", len(TEST_IMGS))
def evaluar_test(cfg):
    preds, ms = predecir(TEST_IMGS, cfg["imgsz"], cfg["augment"], cfg.get("extra"))
    d = contar(preds, TEST_LABELS); del preds
    return d, ms

d_base, _ = evaluar_test(CONFIGS[0])
d_sens, ms_sens = d_base, None
if elegida["nombre"] != CONFIGS[0]["nombre"]:
    d_sens, ms_sens = evaluar_test(elegida)

filas = []
for c in range(NC):
    n = punto(d_base[c], 0.25)
    s = punto(d_sens[c], umbrales[CLASS_NAMES[c]])
    filas.append(dict(clase=CLASS_NAMES[c], etiquetas=d_base[c]["n_gt"],
                      P_normal=round(n["P"], 3), R_normal=round(n["R"], 3), FP_normal=n["FP"], FN_normal=n["FN"],
                      P_sensible=round(s["P"], 3), R_sensible=round(s["R"], 3), FP_sensible=s["FP"], FN_sensible=s["FN"],
                      dR=round(s["R"] - n["R"], 3)))
TEST_TABLA = pd.DataFrame(filas)
print(TEST_TABLA.to_string(index=False))
_t = TEST_TABLA[TEST_TABLA.etiquetas > 0]
print(f"\nTEST | recall medio: normal {_t.R_normal.mean():.3f} -> sensible {_t.R_sensible.mean():.3f}"
      f" | precisión media: normal {_t.P_normal.mean():.3f} -> sensible {_t.P_sensible.mean():.3f}")
print(f"Falsas alarmas totales: normal {TEST_TABLA.FP_normal.sum()} -> sensible {TEST_TABLA.FP_sensible.sum()}"
      f" | daños no detectados: normal {TEST_TABLA.FN_normal.sum()} -> sensible {TEST_TABLA.FN_sensible.sum()}")

## 20) Curvas precisión–recall
Cada línea es una configuración. El punto negro es el perfil normal (0,25) y la estrella roja el sensible elegido, ambos de la configuración correspondiente. Mientras más arriba y a la derecha, mejor.

In [ ]:
import matplotlib.pyplot as plt

def curva(dc):
    if dc["n_gt"] == 0 or len(dc["conf"]) == 0:
        return np.zeros(0), np.zeros(0)
    o = np.argsort(-dc["conf"], kind="stable")
    tpc = np.cumsum(dc["tp"][o]); fpc = np.cumsum(~dc["tp"][o])
    return tpc / dc["n_gt"], tpc / (tpc + fpc)

fig, axs = plt.subplots(1, NC, figsize=(5 * NC, 4.2), sharey=True)
for c, ax in enumerate(np.atleast_1d(axs)):
    for cfg in CONFIGS:
        r_, p_ = curva(RES[cfg["nombre"]]["d"][c])
        ax.plot(r_, p_, lw=1.4, ls="--" if cfg.get("info") else "-", label=cfg["nombre"])
    dn = RES["base 640"]["d"][c]; q = punto(dn, 0.25)
    ax.plot(q["R"], q["P"], "ko", ms=8, label="normal 0.25")
    de = RES[elegida["nombre"]]["d"][c]; q = punto(de, umbrales[CLASS_NAMES[c]])
    ax.plot(q["R"], q["P"], "r*", ms=14, label="sensible elegido")
    ax.axhline(PRECISION_MIN, color="gray", lw=0.8, ls=":")
    ax.set_title(CLASS_NAMES[c]); ax.set_xlabel("recall"); ax.set_xlim(0, 1); ax.set_ylim(0, 1.02); ax.grid(alpha=.3)
np.atleast_1d(axs)[0].set_ylabel("precisión"); np.atleast_1d(axs)[-1].legend(fontsize=7, loc="lower left")
plt.tight_layout(); plt.savefig(CAL_DIR / "curvas_pr_val.png", dpi=130); plt.show()

## 21) Guardar resultados y obtener la línea para tu `.env`
Se guardan las tablas en `calibracion/` (Drive) y se imprime el bloque a pegar en el archivo `.env` del proyecto. **Verificación:** se vuelve a parsear el JSON.

In [ ]:
perfil = dict(conf=round(min(umbrales.values()), 2),
              conf_clase={k: round(v, 2) for k, v in umbrales.items()},
              imgsz=int(elegida["imgsz"]), augment=bool(elegida["augment"]))
linea = "PERFIL_SENSIBLE=" + json.dumps(perfil, separators=(",", ":"))
assert json.loads(linea.split("=", 1)[1]) == perfil
pd.concat({n: t for n, t in TABLAS.items()}, names=["config"]).to_csv(CAL_DIR / "val_por_config.csv")
TEST_TABLA.to_csv(CAL_DIR / "test_normal_vs_sensible.csv", index=False)
(CAL_DIR / "perfil_sensible.env.txt").write_text(linea + "\n")
meta = dict(modelo_sha256=h, precision_min=PRECISION_MIN, min_gain=MIN_GAIN, quick=QUICK, iou_match=IOU_MATCH,
            elegida=elegida["nombre"], ms_img_gpu_colab={n: round(r["ms"], 1) for n, r in RES.items()},
            val_imagenes=len(VAL_IMGS), test_imagenes=len(TEST_IMGS), ultralytics=ultralytics.__version__)
(CAL_DIR / "calibracion_info.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print("Guardado en:", CAL_DIR)
print("\n" + "=" * 70 + "\nPega esta línea en el archivo .env de la aplicación (borra la anterior PERFIL_SENSIBLE si existe):\n")
print(linea)
print("\n" + "=" * 70)
if elegida["augment"] or elegida["imgsz"] > 640:
    print("Nota: esta configuración usa", "TTA" if elegida["augment"] else f"imgsz {elegida['imgsz']}",
          "y será más lenta en tu CPU que la normal; mídelo con la app antes de usarla en videos largos.")